In [1]:
import os
import re
import calendar
import pandas as pd
from datetime import date, datetime


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DIRECTORY = (
    "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data"
)

YEAR_TO_CHECK = 2025

OUTPUT_CSV = os.path.join(
    DIRECTORY,
    f"missing_dates_{YEAR_TO_CHECK}.csv"
)


# ------------------------------------------------------------
# Date patterns supported in filenames
# ------------------------------------------------------------

DATE_PATTERNS = [
    # MM-DD-YYYY, MM_DD_YYYY, MM.DD.YYYY
    re.compile(
        r"(?<!\d)(\d{1,2})[-_.](\d{1,2})[-_.](\d{4})(?!\d)"
    ),

    # YYYY-MM-DD, YYYY_MM_DD, YYYY.MM.DD
    re.compile(
        r"(?<!\d)(\d{4})[-_.](\d{1,2})[-_.](\d{1,2})(?!\d)"
    ),
]


def extract_date_from_filename(filename):
    """
    Extract one calendar date from a filename.

    Returns:
        datetime.date if a valid date is found
        None otherwise
    """
    basename = os.path.basename(filename)

    # MM-DD-YYYY style
    match = DATE_PATTERNS[0].search(basename)

    if match:
        month, day, year = map(int, match.groups())

        try:
            return date(year, month, day)
        except ValueError:
            return None

    # YYYY-MM-DD style
    match = DATE_PATTERNS[1].search(basename)

    if match:
        year, month, day = map(int, match.groups())

        try:
            return date(year, month, day)
        except ValueError:
            return None

    return None


# ------------------------------------------------------------
# Scan directory
# ------------------------------------------------------------

if not os.path.isdir(DIRECTORY):
    raise FileNotFoundError(
        f"Directory does not exist: {DIRECTORY}"
    )

found_dates = set()
files_without_dates = []
invalid_or_other_year = []

for root, _, files in os.walk(DIRECTORY):
    for filename in files:
        full_path = os.path.join(root, filename)

        extracted_date = extract_date_from_filename(filename)

        if extracted_date is None:
            files_without_dates.append(full_path)
            continue

        if extracted_date.year == YEAR_TO_CHECK:
            found_dates.add(extracted_date)
        else:
            invalid_or_other_year.append(
                {
                    "filename": full_path,
                    "extracted_date": extracted_date.isoformat(),
                }
            )


# ------------------------------------------------------------
# Build every expected date in the year
# ------------------------------------------------------------

start_date = date(YEAR_TO_CHECK, 1, 1)
end_date = date(YEAR_TO_CHECK, 12, 31)

all_expected_dates = set(
    pd.date_range(
        start=start_date,
        end=end_date,
        freq="D",
    ).date
)

missing_dates = sorted(all_expected_dates - found_dates)


# ------------------------------------------------------------
# Save missing dates
# ------------------------------------------------------------

missing_df = pd.DataFrame({
    "missing_date": missing_dates,
})

if not missing_df.empty:
    missing_df["missing_date"] = pd.to_datetime(
        missing_df["missing_date"]
    )

    missing_df["weekday"] = (
        missing_df["missing_date"]
        .dt.day_name()
    )

    missing_df["month"] = (
        missing_df["missing_date"]
        .dt.month_name()
    )

    missing_df["missing_date"] = (
        missing_df["missing_date"]
        .dt.strftime("%m-%d-%Y")
    )

missing_df.to_csv(
    OUTPUT_CSV,
    index=False,
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

expected_days = 366 if calendar.isleap(YEAR_TO_CHECK) else 365

print(f"Directory scanned: {DIRECTORY}")
print(f"Year checked: {YEAR_TO_CHECK}")
print(f"Expected dates: {expected_days:,}")
print(f"Unique dates found: {len(found_dates):,}")
print(f"Missing dates: {len(missing_dates):,}")
print(f"Files without recognizable dates: {len(files_without_dates):,}")
print(f"Output saved to: {OUTPUT_CSV}")

if missing_dates:
    print("\nMissing dates:")

    for missing_date in missing_dates:
        print(missing_date.strftime("%m-%d-%Y"))
else:
    print("\nNo missing dates found.")

Directory scanned: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data
Year checked: 2025
Expected dates: 365
Unique dates found: 359
Missing dates: 6
Files without recognizable dates: 2
Output saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_2025.csv

Missing dates:
06-05-2025
06-09-2025
08-10-2025
08-28-2025
09-03-2025
10-27-2025
